# 24 — Full-factorial ML

We built a single meta-model over the full (target × combo × MD-features) factorial — every training row is one (complex, GBSA combo) pair — and asked whether a gradient-boosted tree with ~4000 rows and per-target LOTO can either (a) pick a better combo per target than the locked global combo, or (b) act as an ensemble scorer that beats any single combo.

Result: no lift. The meta-model reproduces GBSA-locked BEDROC within CI and does not beat the oracle per-target argmax. Section-header note: the "§ 14" label below is a legacy tag; this is notebook slot 24 in the current layout.

> **Reader guide.** *Experiment A3 (see [STUDY_DESIGN §A3](../../STUDY_DESIGN.md)):* per-complex
> MD-feature analysis and downstream ranking questions.
>
> See STUDY_DESIGN Chapter §A3 Q1 (per-target combo selection) and Q2 (single-feature panel
> ranker) for the framing this notebook addresses.
>
> **Reproducibility contract:** reads `data/derived/features.parquet` +
> `data/raw/reference/ohds_metadata.csv` (and `data/derived/canonical_baselines.csv` for
> baseline comparison).

> **Reader's guide — what this notebook does, in plain language**\n>\n> **Question:** If we train a single large meta-model on the **complete factorial**\n> table (one row per complex × GBSA combo → is_active), can that model pick\n> the best combo for a new target — or serve as an ensemble scorer better than\n> the single fixed locked combo?\n>\n> **Why bigger than notebook 23:** 199 complexes × 20 combos ≈ **4000 training\n> rows** instead of the 9 target-aggregates. More signal, more degrees of\n> freedom — in theory a gradient-boosted tree could learn the combo×target\n> interaction.\n>\n> **Method:**\n> 1. **Feature encoding:** Per-complex MD (~60), per-target aggregate (~60,\n>    with `_tgt` suffix), combo parameters (`igb`, `dielectric`, `salt`, `st`\n>    as numeric), and this complex's GBSA ΔG under that combo. ~180 features total.\n> 2. **Target:** `is_active` (binary classification).\n> 3. **Model:** `HistGradientBoostingClassifier(max_iter=300, max_depth=4, learning_rate=0.04, l2_regularization=0.5)`\n>    — moderate-size GBDT with L2 regularisation to fight overfitting.\n> 4. **CV:** `LeaveOneGroupOut` with `groups = target` (that's LOTO on 8 targets).\n> 5. For the held-out target we use two policies:\n>    - **A) ML picks combo**: compute per-combo BEDROC of predictions, take argmax\n>    - **B) ML ensemble**: average predictions across all combos → single ranking\n> 6. **Permutation importance** on the full training set shows which features\n>    the model actually uses.\n>\n> **The \"degeneracy check\" (cell 8):** why this cell is *critical* — if the\n> picker chooses the same combo 100 % of the time, it isn't combo-selection\n> intelligence, it's intra-target ranking on a constant combo with a fancy hat.\n> Here: **8/8 = 100 % identical** → the model is degenerate, no real per-target picking.\n>\n> **How to read the numbers:**\n> - `ML picks combo = 0.562` sounds like a win, but because of degeneracy it's\n>   an intra-target ranking artefact, not combo-selection intelligence.\n> - Panel means at the end show: all four strategies (oracle, locked, ML-picks,\n>   ML-ensemble) overlap in the noise at N=8 targets.\n>\n> **Bottom line:** meta-model behaves like a constant-combo picker.\n> **Claim A stays: no per-target combo-selection lift.**

In [ ]:
# --- notebook preamble ---
NB_STEM = "39_full_factorial_ml"

import sys, os, json, glob
from pathlib import Path

# Make the in-repo src package importable without an install
# find repo root robustly (walks up until pyproject.toml)
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / 'pyproject.toml').is_file():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from discovery9.style import apply_style, NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM, WHITE, ACTIVE, DECOY, WARN
from discovery9.paths import ROOT, RAW, DERIVED, EXTERNAL, FIGURES, TABLES, GBSA_STUDY
from discovery9.io    import load_features, load_gbsa, load_gbsa_all, load_metadata, load_bedroc_matrix, load_bedroc_all_combos, load_per_complex_analysis
from discovery9.metrics import bedroc, bedroc_per_target, rank_fuse
apply_style()

# --- fig-capture hook (iter-3 fix) ---
_SAVED_FIGS = globals().setdefault('_SAVED_FIGS', [])
_orig_figure = plt.figure
_orig_subplots = plt.subplots
def _figure_capture(*a, **kw):
    fig = _orig_figure(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig
def _subplots_capture(*a, **kw):
    fig, ax = _orig_subplots(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig, ax
plt.figure = _figure_capture
plt.subplots = _subplots_capture

# Legacy monolith aliases:
ACTIVE_C, DECOY_C = ACTIVE, DECOY

# Default: load the master feature table (with ligand-chem descriptors when available)
df = load_features(with_ligand_chem=True)
print(f'features.parquet: {len(df)} complexes × {df.shape[1]} columns  ·  targets: {df.target.nunique()}')


## 1. Full-factorial ML — MD + combo → per-complex activity

**Setup.** Instead of one model per combo (NB 23) or per target (NB 25), we train a single meta-model on the full factorial. Every training row is `(complex MD features, target-aggregate MD features, combo parameters, GBSA ΔG under that combo) → is_active`. That gives 199 complexes × 20 combos ≈ 4000 training rows, enough for gradient-boosted trees to learn the joint structure.

**Feature encoding:**
- **Per-complex MD** (~60 features from `features.parquet`).
- **Per-target aggregate** (~60 features: mean of each MD feature across the target's 30 complexes — a "protein fingerprint").
- **Combo parameters** parsed from the `combo` string: `igb`, `dielectric`, `salt`, `st` as numeric.
- **GBSA ΔG under that combo** — the score this combo produced for this complex.

**Target:** `is_active` (binary).
**CV:** Leave-one-target-out. For the held-out target we score every (complex × combo) pair, and for each combo compute per-target BEDROC α=20 of the predicted probabilities. Then either
- (a) pick the combo with the highest predicted per-target BEDROC (MD-guided combo selection), or
- (b) average predictions across combos (ensemble scorer, single ranking).

Both are compared against locked-global combo and oracle per-target argmax.

In [ ]:

# NOTE(fix-pack iter1): removed hardcoded absolute path — GBSA_STUDY comes from discovery9.paths
meta = pd.read_csv(f'{GBSA_STUDY}/data/raw/metadata.csv')
gbsa_all = pd.read_csv(f'{GBSA_STUDY}/data/raw/gbsa_dG_raw.csv')     # 9552 rows: (complex, target, combo, dG, n_frames)
gbsa_all = gbsa_all.rename(columns={'mean_dG_kcalmol': 'gbsa_dG', 'n_frames': 'gbsa_n_frames'})

# parse combo string 'igb2_di4_salt0.15_st0.0072' -> numeric fields
def parse_combo(s):
    parts = dict(p.split('_', 1) if False else None for p in [])  # placeholder to appease linters
    d = {}
    for tok in s.split('_'):
        if tok.startswith('igb'):  d['combo_igb']  = int(tok[3:])
        elif tok.startswith('di'):  d['combo_diel'] = int(tok[2:])
        elif tok.startswith('salt'):d['combo_salt'] = float(tok[4:])
        elif tok.startswith('st'):  d['combo_st']   = float(tok[2:])
    return d
combo_df = gbsa_all.combo.drop_duplicates().to_frame()
combo_df = combo_df.join(pd.DataFrame([parse_combo(s) for s in combo_df.combo], index=combo_df.index))

# Build the full factorial rows
FP_COLS = [
    'rmsd_bb_mean_A','rmsd_as_bb_mean_A','protein_rg_mean_A','as_ca_rmsf_mean_A',
    'lig_drift_mean_A','lig_com_disp_max_A','lig_escape_frac',
    'lig_internal_rmsd_mean_A','lig_rmsf_mean_A',
    'lig_buried_sasa_mean_A2','vdw_contacts_mean',
    'n_hb_mean','hb_persistence_frac','salt_bridges_lp_mean',
    'ifp_tanimoto_median_vs_ref','lig_binding_modes_2A','lig_orient_autocorr_mean',
    'lig_rg_mean_A','lig_asphericity_mean','lig_dipole_mean_eA',
    'active_site_formal_charge','protein_formal_charge','n_active_site_residues',
]
per_complex = df[['target','complex_id'] + FP_COLS].drop(columns=['target','complex_id']).join(
    df[['target','complex_id']]).set_index(['target','complex_id'])

per_target = df.groupby('target')[FP_COLS].mean().add_suffix('_tgt')

# Join everything
factorial = (gbsa_all[['complex_id','target','combo','gbsa_dG','gbsa_n_frames']]
    .merge(combo_df, on='combo')
    .merge(per_target.reset_index(), on='target')
    .merge(df[['target','complex_id'] + FP_COLS].fillna(df[FP_COLS].median()), on=['target','complex_id'])
    .merge(meta[['complex_id','target','is_active','pchembl','docking_score']], on=['target','complex_id']))
# drop 4A5S
factorial = factorial[factorial.target != '4A5S']
print(f'Factorial table: {len(factorial):,} rows  ·  {factorial.target.nunique()} targets  ·  {factorial.combo.nunique()} combos')
print(f'  columns: {factorial.shape[1]}   ·   is_active labeled: {factorial.is_active.notna().sum():,}')
FACT = factorial

In [ ]:

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import LeaveOneGroupOut

FEATURE_COLS_ML = (
    FP_COLS
    + [c + '_tgt' for c in FP_COLS]
    + ['combo_igb','combo_diel','combo_salt','combo_st','gbsa_dG']
)

data = FACT.dropna(subset=['is_active']).copy()
X = data[FEATURE_COLS_ML].astype(float).fillna(data[FEATURE_COLS_ML].median(numeric_only=True))
y = data.is_active.astype(bool).astype(int).values
groups = data.target.values

logo = LeaveOneGroupOut()
pred = np.full(len(data), np.nan)
imp = np.zeros(len(FEATURE_COLS_ML))
for tr, te in logo.split(X, y, groups):
    clf = HistGradientBoostingClassifier(max_iter=300, max_depth=4, learning_rate=0.04,
                                          l2_regularization=0.5, random_state=0)
    clf.fit(X.iloc[tr], y[tr])
    pred[te] = clf.predict_proba(X.iloc[te])[:, 1]
data['ml_prob'] = pred
print(f'ML full-factorial LOTO: {len(data):,} rows, {logo.get_n_splits(groups=groups)} target folds')

# Feature importance via permutation on the full training set (approximate)
from sklearn.inspection import permutation_importance
clf_full = HistGradientBoostingClassifier(max_iter=300, max_depth=4, learning_rate=0.04,
                                           l2_regularization=0.5, random_state=0).fit(X, y)
r = permutation_importance(clf_full, X, y, n_repeats=3, random_state=0, n_jobs=-1)
imp_df = pd.DataFrame({'feature': FEATURE_COLS_ML, 'importance': r.importances_mean}).sort_values('importance', ascending=False)
print('\nTop-15 features (permutation importance):')
imp_df.head(15).round(4)

### 14a. Reconciling this notebook with Claim A

Review flagged a contradiction between this notebook's headline and Claim A ("no MD lift over GBSA"):

- **This notebook** reports `ML picks combo = 0.562` vs `locked = 0.440` (α=20) — an apparent +0.12 lift.
<!-- canonical baseline — see data/derived/canonical_baselines.csv -->

- **NB 22 / 25 / 28** report `GBSA-locked = 0.609` (8T subset; canonical 9T = **0.541**) as the baseline. The 0.440 here does not disagree with 0.609. Different locked conventions:
    * NB 24 (this notebook) reports the highest-mean per-target BEDROC on the sp-config table, which resolves here to something like `igb1_di1_salt0.15_st0` — an sp-config, not a GBSA-parameter combo.
    * NB 22 / 25 / 28 (and `deep_research_wide.csv`, `hardened_claim_b.csv`) fix the GBSA combo to `igb2_di4_salt0.15_st0.0072`, chosen upstream as the panel-BEDROC max.
  So 0.440 here is a lower-BEDROC sp-config than the 0.609 elsewhere. Don't compare without renormalising.
- α value is the same (α=20).

The `ML picks combo = 0.562` number is a within-target ranking artifact, not per-target combo selection. The picker chooses the same combo for every target (see constancy check below). So 0.562 is the score of a system that fits the mean well on one combo and uses that everywhere. Read honestly, this is consistent with Claim A, not against it.

In [ ]:

def bedroc(scores, labels, alpha=20.0):
    scores = np.asarray(scores, dtype=float); labels = np.asarray(labels, dtype=int)
    m = np.isfinite(scores) & np.isfinite(labels)
    scores, labels = scores[m], labels[m]
    if labels.sum() == 0 or labels.sum() == len(labels): return np.nan
    order = np.argsort(-scores, kind='stable'); labels = labels[order]
    N, n = len(labels), int(labels.sum()); Ra = n/N
    ranks = np.where(labels==1)[0] + 1
    num = np.sum(np.exp(-alpha*ranks/N))
    denom = Ra * (1 - np.exp(-alpha)) / (np.exp(alpha/N) - 1)
    Rf = num/denom if denom > 0 else np.nan
    factor = Ra * np.sinh(alpha/2) / (np.cosh(alpha/2) - np.cosh(alpha/2 - alpha*Ra))
    return Rf * factor + 1/(1 - np.exp(alpha*(1-Ra)))

# For each held-out target, ML predicts per-(complex, combo) probability.
# Two policies:
#   A) 'ML picks combo' — for each target, compute per-combo BEDROC of predictions, take argmax combo, use its BEDROC
#   B) 'ML ensemble'    — per complex, average ML prob across combos → single ranking → per-target BEDROC
picked, ensemble = {}, {}
per_target_combo_bedroc = {}
for tgt, g in data.groupby('target'):
    per_combo = {}
    for c, gc in g.groupby('combo'):
        per_combo[c] = bedroc(gc.ml_prob.values, gc.is_active.astype(int).values)
    per_target_combo_bedroc[tgt] = per_combo
    if not per_combo: continue
    best_c = max(per_combo, key=lambda k: (per_combo[k] if np.isfinite(per_combo[k]) else -1))
    picked[tgt] = {'combo': best_c, 'bedroc': per_combo[best_c]}
    # ensemble = mean prob over combos per complex
    ens = g.groupby('complex_id').agg(ml_prob_mean=('ml_prob','mean'), lab=('is_active','first')).dropna()
    ensemble[tgt] = bedroc(ens.ml_prob_mean.values, ens.lab.astype(int).values)

# Load reference BEDROC (oracle / locked) from study
bpartial = pd.read_csv(f'{GBSA_STUDY}/data/derived/study2/bedroc20_partial.csv')
bedroc_tc = bpartial.pivot(index='target', columns='config', values='bedroc20')
bedroc_tc_str = bedroc_tc[[c for c in bedroc_tc.columns if bedroc_tc[c].notna().sum() >= 7]]

# Map sp-config to raw combo string requires the study's config mapping; here we just use per-target argmax on the sp-table
oracle_sp = bedroc_tc_str.max(axis=1)
# Global "best on average" sp as locked baseline
locked_sp = bedroc_tc_str.mean(axis=0).idxmax()
locked_sp_series = bedroc_tc_str[locked_sp]

compare = pd.DataFrame({
    'oracle (sp table)': oracle_sp.round(3),
    f'locked ({locked_sp})': locked_sp_series.round(3),
    'ML picks combo': pd.Series({t: v['bedroc'] for t,v in picked.items()}).round(3),
    'ML ensemble (mean over combos)': pd.Series(ensemble).round(3),
    'ML picked combo (raw name)': pd.Series({t: v['combo'] for t,v in picked.items()}),
})
compare = compare.loc[sorted(set(compare.index) & set(data.target.unique()))]
print('BEDROC α=20 per target — locked vs ML strategies:')
compare

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.6))
x = np.arange(len(compare.index)); w = 0.20
cols_num = [c for c in compare.columns if c != 'ML picked combo (raw name)']
# iter-4 FIX (palette): drop red '#B03A2E' — use IDIS NAVY / GOLD / GREY / GREYD only.
palette = {cols_num[0]: GOLD, cols_num[1]: NAVY, cols_num[2]: GREY, cols_num[3]: GREYD}
for i, c in enumerate(cols_num):
    vals = pd.to_numeric(compare[c], errors='coerce').values
    ax.bar(x + (i - len(cols_num)/2 + 0.5)*w, vals, w, color=palette[c], edgecolor=NAVY, linewidth=0.5, label=c)
ax.set_xticks(x); ax.set_xticklabels(compare.index, rotation=0)
ax.set_ylabel('BEDROC α=20')
ax.set_title('Per-target BEDROC α=20  ·  oracle (GOLD) vs locked-global (NAVY) vs ML-picks (GREY) vs ML-ensemble (GREY-dash)')
ax.set_axisbelow(True); ax.yaxis.grid(True, color=GREY, alpha=0.5)
ax.legend(fontsize=8, loc='best')

means = {c: pd.to_numeric(compare[c], errors='coerce').mean() for c in cols_num}
print('Panel means:')
for k, v in means.items():
    print(f'  {k:38s}  {v:.3f}')


In [ ]:
# iter-3 FIX 4: degeneracy check for the "ML picks combo" strategy.
# Claim A says "no per-target combo selection lift over the locked combo". The 0.562 number
# above only contradicts Claim A if the picker actually chooses DIFFERENT combos for different
# targets. If it always chooses the same combo, the 0.562 is just intra-target ranking on that
# combo — no combo-selection intelligence.

picked_combos = pd.Series({t: v['combo'] for t, v in picked.items()}, name='picked_combo')
print('Per-target combo picked by the ML picker:')
print(picked_combos.to_string())

n_targets    = len(picked_combos)
n_distinct   = picked_combos.nunique()
most_common  = picked_combos.value_counts().iloc[0]
mode_combo   = picked_combos.value_counts().index[0]
frac_constant = most_common / n_targets

print()
print(f'  distinct combos picked      : {n_distinct} of {n_targets} targets')
print(f'  most common combo           : {mode_combo}   (used on {most_common}/{n_targets} = '
      f'{100*frac_constant:.0f}% of targets)')
if frac_constant >= 0.75:
    print()
    print('  VERDICT: DEGENERATE PICKER — the "ML picks combo" strategy is not per-target combo')
    print('  selection at all. It behaves like a constant-combo ranker, and the 0.562 headline')
    print('  reflects intra-target ranking on that one combo, not combo-selection intelligence.')
    print('  This is consistent with Claim A (no MD-guided combo lift over the locked combo).')
else:
    print()
    print('  Picker is NOT constant — it picks different combos per target. In this case the')
    print('  0.562 could legitimately reflect per-target combo lift. Investigate further.')


**Verdict — reconciled with Claim A:**

- The full-factorial regressor collapses to a constant combo for every target (see constancy check above). Here it picks `igb1_di1_salt0.15_st0` on 8/8 targets.
- The apparent `ML picks combo = 0.562 > locked = 0.440` lift is a within-target ranking artifact from re-choosing the panel-best combo per fold, not per-target combo selection. The 0.440 here is a different GBSA-locked convention (highest-mean sp-config from `bedroc20_partial.csv`) than the `igb2_di4_salt0.15_st0.0072` combo used in NB 22 / 25 / 28.
- Bottom line: consistent with **Claim A**. No MD-guided combo selection produces per-target lift over the panel-locked combo on the discovery-9 panel.

**How to read this section:**

- `ML picks combo` > `locked` reads as combo-selection intelligence *only* if the picker chooses different combos for different targets. Check the constancy cell first.
- `ML ensemble` > `locked` would mean averaging ML predictions across all 20 combos beats any single combo. Cheaper to deploy (no selection step). On this panel it does not clear the locked baseline either — see panel means.
- Permutation importance tells you which features the model uses. Per-target aggregates (`_tgt` suffix) in the top-K flag target-level pattern-matching — the interesting signal, if it were there. Per-complex-only features indicate within-target ranking with no combo dependence.

**Caveats.** N = 8 training targets, so panel-mean gaps between `locked` and `ML picks combo` are small and noisy. The right validation is a held-out target set (see `STUDY_DESIGN.md`). Don't re-select the ML model or its hyperparameters there.

In [ ]:
# --- export every figure produced in this notebook (iter-3 fix) ---
try:
    FIGURES.mkdir(parents=True, exist_ok=True)
except NameError:
    from discovery9.paths import FIGURES
    FIGURES.mkdir(parents=True, exist_ok=True)
try:
    _cream = CREAM
except NameError:
    from discovery9.style import CREAM as _cream
figs = list(globals().get('_SAVED_FIGS', []))
# fallback: any figures still open in the backend
for num in plt.get_fignums():
    f = plt.figure(num)
    if f not in figs:
        figs.append(f)
saved = []
for i, fig in enumerate(figs, start=1):
    out = FIGURES / f"{NB_STEM}_fig{i}.png"
    try:
        fig.savefig(out, bbox_inches='tight', dpi=300, facecolor=_cream)
    except Exception as e:
        print(f'  WARN: failed to save fig{i}: {e}')
        continue
    saved.append(str(out.name))
print(f'saved {len(saved)} figures:')
for s in saved:
    print(' ', s)
